In [1]:
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12

In [2]:
!pip uninstall -y transformers
!pip install transformers==4.52.4

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 62.3 MB/s eta 0:00:00


In [3]:
!pip uninstall -y torch
!pip install torch==2.6.0

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 37.9 MB/s eta 0:00:00

In [1]:
!pip install numpy==1.26.4

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# 파일 업로드
from google.colab import files
uploaded = files.upload()

Saving final_2.csv to final_2.csv


In [4]:
import pandas as pd
df = pd.read_csv('final_2.csv', encoding = 'cp949')

In [5]:
df.head()

,text,label
0,<SPEAKER_A>안녕! <SPEAKER_B>안녕! <SPEAKER_A>잘 지냈어...,1
1,<SPEAKER_A>이번 주말에 어때? <SPEAKER_A>달콤한 사람아 <SPEA...,1
2,<SPEAKER_A>안녕! <SPEAKER_A>안녕! <SPEAKER_A>좀 늦었는...,1
3,<SPEAKER_A>안녕! <SPEAKER_B>안녕! <SPEAKER_B>어디 갔었...,1
4,"<SPEAKER_A>안녕, 섹시! <SPEAKER_A>잘 지냈어? <SPEAKER_...",1


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1838 entries, 0 to 1837
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1838 non-null   object
 1   label   1838 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 28.8+ KB


In [8]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, f1_score, roc_auc_score
from scipy.special import softmax
import numpy as np
import torch
import re

# 1. Tokenizer 준비 및 스페셜 토큰 추가
tokenizer = AutoTokenizer.from_pretrained("skt/kobert-base-v1")
special_tokens_dict = {'additional_special_tokens': ['<SPEAKER_A>', '<SPEAKER_B>']}
num_added_tokens = tokenizer.add_special_tokens(special_tokens_dict)
print(f"{num_added_tokens} special tokens added to tokenizer.")

# 2. 데이터 분리 및 라벨 정수형 변환
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
train_df['label'] = train_df['label'].astype(int)
val_df['label'] = val_df['label'].astype(int)
train_df.reset_index(drop=True, inplace=True)
val_df.reset_index(drop=True, inplace=True)

print(f"Train 대화 수: {len(train_df)}, Val 대화 수: {len(val_df)}")

# 3. 토크나이징 함수
def tokenize_function(examples):
    texts = examples["text"]
    if isinstance(texts, str):
        texts = [texts]

    input_ids_batch, attention_mask_batch = [], []
    max_len = 512

    for text in texts:
        segments = re.split(r'(<SPEAKER_[AB]>)', text)
        utterances = []
        i = 0
        while i < len(segments) - 1:
            if segments[i].startswith("<SPEAKER_"):
                speaker = segments[i]
                utterance = segments[i + 1].strip()
                full = f"{speaker} {utterance}"
                utterances.append(full)
                i += 2
            else:
                i += 1

        selected = []
        total_tokens = 0
        for utt in reversed(utterances):
            tokens = tokenizer(utt, add_special_tokens=False)["input_ids"]
            if total_tokens + len(tokens) > max_len:
                break
            selected.insert(0, utt)
            total_tokens += len(tokens)

        final_text = " ".join(selected)

        encoded = tokenizer(
            final_text,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=False,  # 여기서 False로 변경
        )

        input_ids_batch.append(encoded["input_ids"])
        attention_mask_batch.append(encoded["attention_mask"])

    return {
        "input_ids": input_ids_batch,
        "attention_mask": attention_mask_batch,
    }

# 4. Huggingface Dataset 객체 생성 및 전처리
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=["text"])
val_dataset = val_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 5. 라벨 정수형 강제 변환
train_dataset = train_dataset.map(lambda x: {"label": int(x["label"])})
val_dataset = val_dataset.map(lambda x: {"label": int(x["label"])})

# 6. 텐서 포맷 지정 (token_type_ids 제거)
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# 7. Data collator 준비
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 8. 모델 준비 및 토큰 임베딩 리사이즈
model = AutoModelForSequenceClassification.from_pretrained("skt/kobert-base-v1", num_labels=2)
model.resize_token_embeddings(len(tokenizer))
print(f"Model embedding size resized to {len(tokenizer)}.")

# 9. 평가 지표 함수
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)
    preds = np.argmax(probs, axis=1)

    recall = recall_score(labels, preds)
    f1 = f1_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "recall": recall,
        "f1": f1,
        "auc": auc,
    }

# 10. 학습 설정
training_args = TrainingArguments(
    output_dir='./results_kobert_final_2',
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_dir='./logs_kobert_final_2',
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    greater_is_better=True,
    report_to="wandb",
)

# 11. Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


2 special tokens added to tokenizer.
Train 대화 수: 1470, Val 대화 수: 368


Map:   0%|          | 0/1470 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

Map:   0%|          | 0/1470 [00:00<?, ? examples/s]

Map:   0%|          | 0/368 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/369M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at skt/kobert-base-v1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model embedding size resized to 8004.


/tmp/ipython-input-8-2433855686.py:141: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [9]:
# 11️⃣ 학습 시작
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: seongmin2053 (seongmin2053-dong-eui-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [13]:
model.save_pretrained('/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run5')
tokenizer.save_pretrained('/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run5')

('/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4/tokenizer_config.json',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4/special_tokens_map.json',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4/spiece.model',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4/added_tokens.json',
 '/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4/tokenizer.json')

### 평가

In [14]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

device = torch.device("cpu")  # 필요시 "cuda"로 변경

model_path = "/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.resize_token_embeddings(len(tokenizer))  # 안전하게 재조정
model.to(device)
model.eval()

def predict_grooming_proba(text, model, tokenizer, max_len=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        padding="max_length",
        add_special_tokens=True,
        return_token_type_ids=False  # 학습 때와 맞춤
    )
    # 학습 때 token_type_ids 안 쓰니까 입력에서 빼기
    inputs = {k: v.to(device) for k, v in inputs.items() if k != "token_type_ids"}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
        pred = probs.argmax()

    return {
        "predicted_label": int(pred),
        "probabilities": {
            "0 (일반 대화)": float(probs[0]),
            "1 (그루밍 의심)": float(probs[1]),
        },
    }

# 대화 루프

history = []

while True:
    user_input = input("💬 대화를 입력하세요 (exit 입력 시 종료): ").strip()
    if user_input.lower() == "exit":
        history.clear()
        break

    history.append(user_input)
    if len(history) > 30:
        history.pop(0)

    full_text = " ".join(history)
    result = predict_grooming_proba(full_text, model, tokenizer)

    for i, conversation in enumerate(history):
        print(f"[대화 히스토리] ({i}) : {conversation}")
    print('=' * 50)
    print(f"[예측 라벨] ➤ {result['predicted_label']}")
    print("[확률 분포]")
    for k, v in result["probabilities"].items():
        print(f" - {k}: {v:.4f}")
    print('=' * 50)


💬 대화를 입력하세요 (exit 입력 시 종료): <SPEAKER_A>안녕
[대화 히스토리] (0) : <SPEAKER_A>안녕
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.9548
 - 1 (그루밍 의심): 0.0452
💬 대화를 입력하세요 (exit 입력 시 종료): <SPEAKER_B>하이
[대화 히스토리] (0) : <SPEAKER_A>안녕
[대화 히스토리] (1) : <SPEAKER_B>하이
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.9539
 - 1 (그루밍 의심): 0.0461
💬 대화를 입력하세요 (exit 입력 시 종료): <SPEAKER_A>몇살이야?
[대화 히스토리] (0) : <SPEAKER_A>안녕
[대화 히스토리] (1) : <SPEAKER_B>하이
[대화 히스토리] (2) : <SPEAKER_A>몇살이야?
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.9546
 - 1 (그루밍 의심): 0.0454
💬 대화를 입력하세요 (exit 입력 시 종료): <SPEAKER_B>13살
[대화 히스토리] (0) : <SPEAKER_A>안녕
[대화 히스토리] (1) : <SPEAKER_B>하이
[대화 히스토리] (2) : <SPEAKER_A>몇살이야?
[대화 히스토리] (3) : <SPEAKER_B>13살
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.9536
 - 1 (그루밍 의심): 0.0464
💬 대화를 입력하세요 (exit 입력 시 종료): <SPEAKER_A>나는 28살이야
[대화 히스토리] (0) : <SPEAKER_A>안녕
[대화 히스토리] (1) : <SPEAKER_B>하이
[대화 히스토리] (2) : <SPEAKER_A>몇살이야?
[대화 히스토리] (3) : <SPEAKER_B>13살
[대화 히스토리] (4) : <SPEAKER_A>나는 28살이야
[예측 라벨] ➤ 0
[확률 분포]
 - 0 (일반 대화): 0.9524
 - 1 (그루밍 의심): 0.047

KeyboardInterrupt: Interrupted by user

In [6]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import torch.nn.functional as F

device = torch.device("cpu")  # 필요시 "cuda"로 변경

model_path = "/content/drive/MyDrive/디스부_최종프로젝트/Detection/대화/KoBERT_run4"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.resize_token_embeddings(len(tokenizer))  # 안전하게 재조정
model.to(device)
model.eval()

def predict_grooming_proba(text, model, tokenizer, max_len=512):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_len,
        padding="max_length",
        add_special_tokens=True,
    )
    if "token_type_ids" not in inputs:
        inputs["token_type_ids"] = torch.zeros_like(inputs["input_ids"])
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=-1).squeeze().cpu().numpy()
        pred = probs.argmax()

    return {
        "predicted_label": int(pred),
        "probabilities": {
            "0 (일반 대화)": float(probs[0]),
            "1 (그루밍 의심)": float(probs[1]),
        },
    }


In [8]:
# 대화 루프

history = []

while True:
    user_input = input("💬 대화를 입력하세요 (exit 입력 시 종료): ").strip()
    if user_input.lower() == "exit":
        history.clear()
        break

    # ✅ 히스토리에 저장 (최대 30개 유지)
    history.append(user_input)
    if len(history) > 30:
        history.pop(0)  # 가장 오래된 발화 제거

    # ✅ 누적된 대화를 하나의 입력으로 구성
    full_text = " ".join(history)

    # ✅ 예측
    result = predict_grooming_proba(full_text, model, tokenizer)
    for i, conversation in enumerate(history):
        print(f"[대화 히스토리] ({i}) : {conversation}")
    print('=' * 50)
    print(f"[예측 라벨] ➤ {result['predicted_label']}")
    print("[확률 분포]")
    for k, v in result["probabilities"].items():
        print(f" - {k}: {v:.4f}")
    print('=' * 50)

💬 대화를 입력하세요 (exit 입력 시 종료): <SPEAKER_A>안녕


IndexError: index out of range in self